### Experiment tracking kyon?

Abhi tak tumhare experiments ke numbers CSV files aur notebook outputs mein bikhre hain (`cv_ensembles_day83.csv`, `tuning_results_day84.csv` ...). Teen mahine baad “yeh 0.78 wala run kaunse parameters, kaunse data aur kaunse code se aaya tha?” ka jawab dhoondhna mushkil hoga. Experiment tracking har run ke saath yeh sab ek jagah likh deta hai:

### MLflow ka concept

| MLflow ka concept | Matlab | Hamare project mein |
| :--- | :--- | :--- |
| **Experiment** | Runs ka group | `churn-day91-intro` |
| **Run** | Ek baar ka training/CV | `"LogReg, C=0.1"` |
| **Params** | Settings jo tum chunte ho | `hp_C`, `cv_n_splits`, data fingerprint |
| **Metrics** | Natije (step ke saath bhi) | `cv_roc_auc_mean`, fold-wise AUC |
| **Artifacts** | Files | feature list, plots |
| **Tags** | Label/context | `git_commit`, `git_dirty`, `stage` |

**MLflow vs Weights & Biases:** MLflow open-source hai, laptop par chalta hai, account nahi chahiye, aur data tumhari machine par rehta hai. W&B ke dashboards behtar hain, lekin cloud account chahiye aur data bahar jaata hai. Is project ke liye MLflow chuna hai.

**Rules:**

1. Tracking mein sirf CV metrics jaayenge. Test set ka koi number log nahi hoga, taaki “kaunsa run best hai” ka faisla test dekh kar na bane (pre-registration wali baat). Isko code mein guard se enforce karenge.
2. MLflow ek development tool hai. Wo `requirements.txt` (app) mein nahi, `requirements-dev.txt` mein jaayega, aur uska store (`mlflow_store/`) Git mein nahi jaayega.
3. Tracking record karti hai, faisla nahi karti. Final model ka faisla abhi bhi `reports/final_model_choice_day84.md` mein hai.

### 1. Setup

In [1]:
import os
import platform
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Project root auto-detect -------------------------------------------------
# FIX 1: pehle 'data/processed' dhoondhte the. Wo folder .gitignore mein hai (har machine par alag),
# isliye detection upar ke kisi bhi folder (jaise C:\Users\...\OneDrive) par ruk sakta tha.
# Ab repo ki apni tracked file ko marker banate hain, aur sabse NAZDEEKI folder chunte hain.
MARKER = Path('src') / 'tracking' / 'mlflow_utils.py'
_start = Path.cwd().resolve()
project_root = next((p for p in [_start, *_start.parents] if (p / MARKER).is_file()), None)
if project_root is None:
    raise FileNotFoundError(f'Repo root nahi mila ({MARKER} kisi parent folder mein nahi). '
                            f'Notebook ko repo ke andar se kholo. Abhi cwd: {_start}')
os.chdir(project_root)
# FIX 2: insert(0) - append karne par upar ke kisi aur 'src' package se import shadow ho sakta tha.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
DATA_DIR = project_root / 'data' / 'processed'

import mlflow
import pandas as pd
import sklearn
from matplotlib.figure import Figure
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.features.build_features import NUM_V2
from src.tracking.mlflow_utils import (dataset_fingerprint, file_sha256, git_info,
                                       log_cv_metrics, setup_tracking)

exp_id = setup_tracking('churn-day91-intro')
print('project_root :', project_root)
print('mlflow       :', mlflow.__version__, '| experiment id:', exp_id)
print('tracking uri :', mlflow.get_tracking_uri())

project_root : C:\Users\rajat\OneDrive\Desktop\ecommerce-churn-project
mlflow       : 3.17.2 | experiment id: 1
tracking uri : sqlite:///C:/Users/rajat/OneDrive/Desktop/ecommerce-churn-project/mlflow_store/mlflow.db


### 2. Ek “hello” run (sirf UI dekhne ke liye)

In [2]:
with mlflow.start_run(run_name='hello'):
    mlflow.log_param('purpose', 'UI dekhne ke liye')
    mlflow.set_tag('stage', 'demo')          # FIX: 'cv' runs se alag pehchaan (Step 6 ka filter isi par tikta hai)
    for step in range(5):
        log_cv_metrics({'cv_demo_curve': 1 - 0.5 ** (step + 1)}, step=step)
    mlflow.log_dict({'note': 'artifact example'}, 'note.json')

### 3. Data aur model

**Prerequisite:** `data/processed/X_train_v2.csv` Notebook 83 (cell 15) banata hai aur `data/` Git mein nahi hota. Nahi mili to code `app_artifacts/customer_features_app.csv` se wahi train rows utha leta hai (aur `data_source` param mein likh deta hai).

In [3]:
FEATURE_COLS = [*NUM_V2, 'Region_Rest of World', 'Region_UK']      # make_xy(feat, NUM_V2) ka exact order
X_PATH = DATA_DIR / 'X_train_v2.csv'                               # Notebook 83 (cell 15) banata hai
Y_PATH = DATA_DIR / 'y_train.csv'                                  # Notebook 67 banata hai
APP_TABLE = project_root / 'app_artifacts' / 'customer_features_app.csv'   # Git mein tracked hai

if X_PATH.exists() and Y_PATH.exists():
    DATA_SOURCE, DATA_FILE = 'data/processed', X_PATH
    X = pd.read_csv(X_PATH).astype(float)
    y = pd.read_csv(Y_PATH).iloc[:, 0]
elif APP_TABLE.exists():
    # FIX 3: data/ gitignored hai, fresh clone par nahi hota. Tab wahi train rows app table se lo.
    # (Row order alag hota hai, to folds thode alag; isliye tracking mein data_source bhi log hoga.)
    DATA_SOURCE, DATA_FILE = 'app_artifacts-fallback', APP_TABLE
    tbl = pd.read_csv(APP_TABLE)
    tr = tbl[tbl['split'] == 'train'].reset_index(drop=True)
    X = tr[FEATURE_COLS].astype(float)
    y = tr['observed_churn'].rename('Churned')
    print('WARNING: data/processed mein X_train_v2.csv nahi mili -> app_artifacts se train rows use ho rahe hain.')
else:
    raise FileNotFoundError(
        f'Training data nahi mila.\n  Dhoondha: {X_PATH}\n  Fallback: {APP_TABLE}\n'
        'Fix: Notebook 83 (cell 15) chalao taaki data/processed/X_train_v2.csv bane, '
        'ya repo ka app_artifacts/ folder wapas rakho.')

# Data sanity checks (galat/adhoora data ka pata yahin chal jaaye)
missing = [c for c in NUM_V2 if c not in X.columns]
assert not missing, f'X mein ye columns nahi hain: {missing}'
assert len(X) == len(y), (len(X), len(y))
assert not X.isna().any().any(), 'X mein NaN hai'
assert set(y.unique()) <= {0, 1}, 'y binary (0/1) nahi hai'
print(f'data source: {DATA_SOURCE} | X: {X.shape} | churn rate: {y.mean():.4f}')

prep = ColumnTransformer([('s', StandardScaler(), NUM_V2)], remainder='passthrough')
model = Pipeline([('prep', prep), ('m', LogisticRegression(max_iter=5000, class_weight='balanced'))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = {'roc_auc': 'roc_auc', 'ap': 'average_precision', 'recall': 'recall', 'f1': 'f1'}

data source: data/processed | X: (4226, 21) | churn rate: 0.5663


Yeh bilkul Day 83 ka LogReg v2 setup hai: wahi folds, wahi model.

### 4. Ek reusable “tracked CV run” function

In [4]:
def run_cv_experiment(run_name, model, extra_tags=None):
    est = model.named_steps['m']
    # params model se nikaale, haath se nahi likhe: warna logged params aur asli model alag ho sakte hain
    params = {'model': type(est).__name__, **{f'hp_{k}': v for k, v in est.get_params().items()}}

    with mlflow.start_run(run_name=run_name) as run:
        # FIX 4: cv_n_splits / cv_seed pehle haath se (5, 42) likhe the; ab seedha cv object se.
        mlflow.log_params({**params, 'cv_n_splits': cv.get_n_splits(), 'cv_seed': cv.random_state,
                           'scoring_primary': 'roc_auc', 'n_features': X.shape[1],
                           'data_source': DATA_SOURCE,
                           'data_file_sha': file_sha256(DATA_FILE)[:12]})
        mlflow.log_params({f'data_{k}': v for k, v in dataset_fingerprint(X, y).items()})
        mlflow.set_tags({**git_info(), 'stage': 'cv', 'test_set_used': 'false',
                         'sklearn': sklearn.__version__, 'python': platform.python_version(),
                         **(extra_tags or {})})

        # error_score='raise': koi fold fail ho to chup-chaap NaN log na ho, seedha error aaye
        res = cross_validate(model, X, y, cv=cv, scoring=SCORING, error_score='raise')

        # sklearn ke 'test_*' keys CV ke validation folds hain, held-out test set nahi.
        # Guard ke liye naam 'cv_' rakhte hain.
        summary = {}
        for name in SCORING:
            scores = res[f'test_{name}']
            summary[f'cv_{name}_mean'] = float(scores.mean())
            summary[f'cv_{name}_std'] = float(scores.std())
        log_cv_metrics(summary)
        fold_auc = res['test_roc_auc']
        for i, v in enumerate(fold_auc):
            log_cv_metrics({'cv_fold_roc_auc': float(v)}, step=i)

        fig = Figure(figsize=(4.5, 3))
        ax = fig.subplots()
        ax.bar(range(1, len(fold_auc) + 1), fold_auc, color='steelblue')   # FIX 4: range(1, 6) hardcoded tha
        ax.set_ylim(0.5, 1.0)
        ax.set_xlabel('CV fold')
        ax.set_ylabel('ROC-AUC')
        ax.set_title('Fold-wise AUC (axis starts at 0.5 = chance)')
        fig.tight_layout()
        mlflow.log_figure(fig, 'fold_roc_auc.png')
        mlflow.log_dict({'features': list(X.columns)}, 'features.json')
    return run.info.run_id, summary

### 5. Runs chalao

In [5]:
rid1, s1 = run_cv_experiment('logreg_v2_balanced_C1', model)
print({k: round(v, 4) for k, v in s1.items()})

# Sanity: tracked number wahi hai jo Day 83 mein report hua tha?
day83 = pd.read_csv(project_root / 'reports' / 'cv_ensembles_day83.csv', index_col=0)
ref_auc = float(day83.loc['LogReg v2', 'roc_auc'])
# Day 83 CSV 4 decimal par round hai. Fallback data mein rows ka order alag hota hai, isliye wahan tolerance thodi badi.
TOL = 1e-3 if DATA_SOURCE == 'data/processed' else 3e-3
diff = abs(s1['cv_roc_auc_mean'] - ref_auc)
print(f"Day 83 LogReg v2 AUC: {ref_auc} | tracked: {s1['cv_roc_auc_mean']:.4f} | diff: {diff:.4f} (tol {TOL})")
assert diff < TOL, 'Tracked AUC Day 83 se match nahi karta: folds ya features alag hain (tracking se pehle ki bug).'

# Wahi run dobara: metrics identical aane chahiye (reproducibility)
rid2, s2 = run_cv_experiment('logreg_v2_balanced_C1_rerun', model)
print('max metric diff between identical runs:', max(abs(s1[k] - s2[k]) for k in s1))

# Ek alag setting
rid3, s3 = run_cv_experiment('logreg_v2_balanced_C0.1', clone(model).set_params(m__C=0.1))
print('C=0.1 AUC:', round(s3['cv_roc_auc_mean'], 4))

{'cv_roc_auc_mean': 0.796, 'cv_roc_auc_std': 0.0118, 'cv_ap_mean': 0.8204, 'cv_ap_std': 0.0117, 'cv_recall_mean': 0.7464, 'cv_recall_std': 0.0148, 'cv_f1_mean': 0.7534, 'cv_f1_std': 0.01}
Day 83 LogReg v2 AUC: 0.796 | tracked: 0.7960 | diff: 0.0000 (tol 0.001)
max metric diff between identical runs: 0.0
C=0.1 AUC: 0.7961


Do checks yahan sabse zaroori hain. Pehla: tracking ne wahi number diya jo tumne Day 83 mein report kiya tha (agar fark aaye, to ya folds ya features alag hain, aur wo tracking se pehle ki bug hai). Doosra: same run dobara chalane par metrics same aate hain.

### 6. Runs ko code se query karo

In [6]:
# FIX 5: sirf 'cv' stage ke runs (hello/demo run mein cv_roc_auc_mean hota hi nahi tha) aur
# khaali/adhoore result par KeyError ki jagah saaf message.
runs = mlflow.search_runs(experiment_names=['churn-day91-intro'],
                          filter_string="tags.stage = 'cv'",
                          order_by=['metrics.cv_roc_auc_mean DESC'])
cols = ['tags.mlflow.runName', 'metrics.cv_roc_auc_mean', 'metrics.cv_roc_auc_std',
        'params.hp_C', 'tags.git_commit', 'tags.git_dirty']
if runs.empty:
    print("Abhi koi 'cv' run nahi hai. Pehle upar wala 'Runs chalao' cell chalao.")
else:
    print(runs.reindex(columns=cols).round(4).to_string(index=False))

        tags.mlflow.runName  metrics.cv_roc_auc_mean  metrics.cv_roc_auc_std params.hp_C tags.git_commit tags.git_dirty
    logreg_v2_balanced_C0.1                   0.7961                  0.0120         0.1         d212d16           True
logreg_v2_balanced_C1_rerun                   0.7960                  0.0118         1.0         d212d16           True
      logreg_v2_balanced_C1                   0.7960                  0.0118         1.0         d212d16           True


### 7. UI tour (browser mein khud karo)

Terminal mein repo root se `mlflow ui --backend-store-uri sqlite:///mlflow_store/mlflow.db` chalao, phir:

1. Runs table mein “Columns” se `cv_roc_auc_mean`, `hp_C`, `git_dirty` dikhao.
2. Search box mein try karo: `tags.test_set_used = 'false'` aur `metrics.cv_roc_auc_mean > 0.5`.
3. `logreg_v2_balanced_C1` kholo: Parameters, Metrics (`cv_fold_roc_auc` ka step chart), Artifacts (`fold_roc_auc.png`, `features.json`), Tags.
4. Teeno runs select karke Compare dabao. C=1 aur C=0.1 ka fark parameters aur metrics dono mein dikhega.
5. `git_dirty` tag dekho. Agar True hai, to notebook ya code mein uncommitted badlav tha, aur commit hash poora sach nahi batata.

### 8. Dhyan rakhne ki baatein

- MLflow tracking after-the-fact bhi ho sakti hai, par uska matlab “pehle se tay” nahi hota. Day 83-84 ke experiments pehle ho chuke hain. Kal unhe MLflow mein dobara log karenge. Wo sirf record ban jaayega. Final model ka faisla jo Day 84 mein commit hua, wahi sach rahega.
- Metric naam ka niyam: CV ke natije `cv_` se shuru honge. Sklearn ke `test_roc_auc` jaise keys CV ke validation folds ke hain (held-out test set ke nahi), isliye naam badalna zaroori hai. Guard galat naam par error dega.
- Store backup: `mlflow_store/` Git mein nahi jaayega, aur wo reproducible nahi hai (runs dobara chalane padenge). Is liye jo numbers README mein jaane hain, unhe `reports/` ki CSV/MD mein bhi save karo.